# Short-term memory for agents
An agent that forgets everything when a run ends is not much of an agent. In this notebook the agent gets **memory**: first I show the problem (it forgets), then I fix it with a **checkpointer** and a **thread id**, and I check what really happens inside the message list.

Only `GOOGLE_API_KEY` is needed (from `.env`, Colab Secrets or pasted below). No OpenAI key is used.

**Idea:** the memory of a chatbot is not a property of the model. It lives in the agent's **state** (the list of messages). By default the state is thrown away after each run.

**Without memory:** every run starts from nothing.
```
Run 1:  [question 1]  -->  Agent  -->  [question 1, answer 1]     (state thrown away)
Run 2:  [question 2]  -->  Agent  -->  [question 2, answer 2]     (starts from nothing again)
```

**With a checkpointer:** a snapshot of the state is saved at the end of every run, under a **thread id**.
```
Run 1 (thread "1"):  [question 1]                --> Agent --> [q1, a1]          --> snapshot saved
Run 2 (thread "1"):  snapshot loaded + [q2]      --> Agent --> [q1, a1, q2, a2]  --> snapshot saved
Run 3 (thread "2"):  nothing to load + [q3]      --> Agent --> [q3, a3]          (a different conversation)
```
- **State** = what the agent tracks during a run (the messages).
- **Checkpointer** = saves a snapshot of the state after each run.
- **Thread id** = the label that groups runs into one conversation.

## Setup

In [1]:
!pip install -q langchain langgraph langchain-google-genai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 14.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()  # local run: reads the .env file (returns False if none is found)

KEYS = ("GOOGLE_API_KEY",)

# Colab run: read the keys from the Secrets panel (key icon in the left sidebar)
try:
    from google.colab import userdata
    for key in KEYS:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

def is_real(value):
    # empty values and template placeholders ("your_..._here") count as missing
    return bool(value) and not value.startswith("your_")

# Still missing? Paste it here (hidden input).
for key in KEYS:
    if not is_real(os.getenv(key)):
        try:
            value = getpass(f"{key} is missing - paste it (Enter to skip): ").strip()
        except Exception:
            value = ""
        if value:
            os.environ[key] = value

for key in KEYS:
    print(f"{key}: {'set' if is_real(os.getenv(key)) else 'MISSING'}")

# Tracing with a placeholder LangSmith key only produces 401/403 warnings: switch it off
if not is_real(os.getenv("LANGSMITH_API_KEY")):
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing: off (no valid LANGSMITH_API_KEY)")

GOOGLE_API_KEY is missing - paste it (Enter to skip): ··········
GOOGLE_API_KEY: set
LangSmith tracing: off (no valid LANGSMITH_API_KEY)


In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pprint import pprint

MODEL = "google_genai:gemini-3.1-flash-lite"

def show(title, text):
    print(f"\n{'=' * 12} {title} {'=' * 12}\n{text}")

def show_messages(messages):
    """Print EVERY message of a list, not just the final answer."""
    for i, m in enumerate(messages):
        print(f"[{i}] {type(m).__name__}")
        print("    content   :", repr(m.text))

def run(agent, text, config=None):
    """Ask one question, print how many messages the agent holds and the final answer."""
    response = agent.invoke({"messages": [HumanMessage(text)]}, config)
    show("Question", text)
    show("Messages in the state", len(response["messages"]))
    show("Answer", response["messages"][-1].text)
    return response

## Part 1: The problem, an agent with no memory
### Task 1: break it first
Tell the agent something about yourself, then ask about it in a **second run**. Watch it forget. The fix means more once you have seen the problem.

> **`agent.invoke({"messages": [...]})`**: one run. When the run ends, the state (the messages) is thrown away.

In [4]:
agent_no_memory = create_agent(model=MODEL)

run(agent_no_memory, "Hello my name is Fatima and my favourite colour is green")


============ Question ============
Hello my name is Fatima and my favourite colour is green

============ Messages in the state ============
2

============ Answer ============
Hello Fatima! It’s lovely to meet you. Green is such a beautiful and refreshing colour—it reminds me of nature, forests, and new beginnings.

Since green is your favorite, do you have a specific shade you like best? For example, are you more of a fan of deep forest green, bright lime, or perhaps a soft sage?


{'messages': [HumanMessage(content='Hello my name is Fatima and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='009227b4-95f3-4d3d-9f2b-f70d88f4b698'),
  AIMessage(content=[{'type': 'text', 'text': 'Hello Fatima! It’s lovely to meet you. Green is such a beautiful and refreshing colour—it reminds me of nature, forests, and new beginnings.\n\nSince green is your favorite, do you have a specific shade you like best? For example, are you more of a fan of deep forest green, bright lime, or perhaps a soft sage?', 'extras': {'signature': 'EnEKbwFpFH0TYi/KUK4HwS1uHOHXxWIulqwLPoBCoz5VlXZHDtxUrS6xpccDPWWbegZ2QX2ZVS2dYqO2tIKUtNZZhHP6uVMF6MncuuuWSVVnU5lyylcjDtrY8qF45VODzTdLSznWJgaMC3/+OJ5Xj2tXzg=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d554-275d-7d53-b114-7eeb54273551-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={

In [5]:
response = run(agent_no_memory, "What's my favourite colour?")


============ Question ============
What's my favourite colour?

============ Messages in the state ============
2

============ Answer ============
I don’t know what your favorite color is yet! You haven't told me. 

If you tell me what it is, I’ll be sure to remember it for next time. So, what is it?


Read the result: **"Messages in the state: 2"**. The second run holds only my new question and its answer. The first exchange is gone, so the agent has no idea who I am. It does not admit a problem, it just cannot know.

**The model is not to blame.** In notebook 1_1 the agent took a *list* of messages. If I give it the earlier messages myself, it answers correctly. That is exactly what memory automates:

In [6]:
from langchain.messages import AIMessage

# Memory by hand: I pass the whole conversation every time
history = [
    HumanMessage("Hello my name is Fatima and my favourite colour is green"),
    AIMessage("Nice to meet you, Fatima! Green is a lovely colour."),
    HumanMessage("What's my favourite colour?"),
]

response = agent_no_memory.invoke({"messages": history})
show("Answer with the history given by hand", response["messages"][-1].text)


============ Answer with the history given by hand ============
Your favourite colour is green!


## Part 2: The fix, a checkpointer
Same agent, plus one argument. The checkpointer saves a snapshot of the state at the end of every run. Runs that share the same **thread id** share the same conversation.

> **`InMemorySaver()`**: a checkpointer that keeps the snapshots in the computer's memory (RAM).  **`checkpointer=`**: the argument of `create_agent` that switches memory on.  **`config = {"configurable": {"thread_id": "1"}}`**: tells the agent which conversation this run belongs to.

In [7]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=MODEL,
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "1"}}   # the conversation label

run(agent, "Hello my name is Fatima and my favourite colour is green", config)


============ Question ============
Hello my name is Fatima and my favourite colour is green

============ Messages in the state ============
2

============ Answer ============
Hello Fatima! It’s lovely to meet you. Green is a wonderful choice—it's such a calming and refreshing color, often reminding people of nature and new beginnings.

Do you have a specific shade of green you like best, like a deep forest green or a bright lime?


{'messages': [HumanMessage(content='Hello my name is Fatima and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='a7a5620e-1487-415a-bd2f-9613a9f5236e'),
  AIMessage(content=[{'type': 'text', 'text': "Hello Fatima! It’s lovely to meet you. Green is a wonderful choice—it's such a calming and refreshing color, often reminding people of nature and new beginnings.\n\nDo you have a specific shade of green you like best, like a deep forest green or a bright lime?", 'extras': {'signature': 'EnEKbwFpFH0Titgf6hTuQnaeuIgXYPJsSIRx5wCsLCaWxL69de/0TKbzbtVFRU03VsHDmF28TUzjcUH/sr99YjfgAeaB6EHBqv820iAORHzgA4SOXLinMkjOmYcnUwC88T0YUmc/6FuDxfbIDlsS920GHg=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d554-923a-7ed2-aca5-1b0c5a034f52-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 59, 'total_tok

In [8]:
response = run(agent, "What's my favourite colour?", config)


============ Question ============
What's my favourite colour?

============ Messages in the state ============
4

============ Answer ============
Your favourite colour is **green**!


Now the second run says **"Messages in the state: 4"** and the agent knows the answer.

### Task 2: check the message list, not just the reply
Print every message. The earlier conversation is sitting inside the list:
- `[0]` and `[1]`: the **first run** (my introduction and the answer to it), loaded from the snapshot.
- `[2]` and `[3]`: the **second run** (my new question and the final answer).

This is the list shape from notebook 1_1. It is not an accident of API design: it is what makes memory possible.

In [9]:
show_messages(response["messages"])

[0] HumanMessage
    content   : 'Hello my name is Fatima and my favourite colour is green'
[1] AIMessage
    content   : "Hello Fatima! It’s lovely to meet you. Green is a wonderful choice—it's such a calming and refreshing color, often reminding people of nature and new beginnings.\n\nDo you have a specific shade of green you like best, like a deep forest green or a bright lime?"
[2] HumanMessage
    content   : "What's my favourite colour?"
[3] AIMessage
    content   : 'Your favourite colour is **green**!'


The snapshot itself can also be read from the agent:

> **`agent.get_state(config)`**: returns the saved snapshot of a thread. The messages are in `.values["messages"]`.

In [10]:
snapshot = agent.get_state(config)

print("Messages saved for thread '1':", len(snapshot.values["messages"]))

Messages saved for thread '1': 4


## Part 3: The thread id
### Task 3: change the thread id and ask again
Same agent, same checkpointer, same question, but a **new thread id**. It is a different conversation, so there is nothing to load.

In [11]:
config_2 = {"configurable": {"thread_id": "2"}}

run(agent, "What's my favourite colour?", config_2)


============ Question ============
What's my favourite colour?

============ Messages in the state ============
2

============ Answer ============
I don’t know what your favorite color is yet! Since I’m an AI, I don’t have access to your personal history unless you’ve told me before in this specific conversation.

However, I’d love to guess—is it **blue**? (Statistically, it’s the most popular favorite color!)

If I’m wrong, tell me what it is and I’ll make sure to remember it.


{'messages': [HumanMessage(content="What's my favourite colour?", additional_kwargs={}, response_metadata={}, id='9d5f4c88-6950-41ea-8c6c-62c3784aa718'),
  AIMessage(content=[{'type': 'text', 'text': 'I don’t know what your favorite color is yet! Since I’m an AI, I don’t have access to your personal history unless you’ve told me before in this specific conversation.\n\nHowever, I’d love to guess—is it **blue**? (Statistically, it’s the most popular favorite color!)\n\nIf I’m wrong, tell me what it is and I’ll make sure to remember it.', 'extras': {'signature': 'EnEKbwFpFH0TotxTg96hQsDUIxG+nFk5Gy+jOd2cUVoWko5NwTl5p3Q1C3d67HZHJtDGiHDWqrZBSKLUPkkWkdieC3lBDm6jjmDjxovoiH4ldRN+XjHD9a8VdeBm2sN9ga3utMYDLdgdKfV7DMxbRo6zyA=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d554-a32c-7e21-ba75-9fbdc83059c8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'

The memory vanished, because thread `"2"` has no snapshot. Nothing was deleted: go back to thread `"1"` and it is all still there.

In [12]:
run(agent, "What's my name?", config)


============ Question ============
What's my name?

============ Messages in the state ============
6

============ Answer ============
Your name is **Fatima**!


{'messages': [HumanMessage(content='Hello my name is Fatima and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='a7a5620e-1487-415a-bd2f-9613a9f5236e'),
  AIMessage(content=[{'type': 'text', 'text': "Hello Fatima! It’s lovely to meet you. Green is a wonderful choice—it's such a calming and refreshing color, often reminding people of nature and new beginnings.\n\nDo you have a specific shade of green you like best, like a deep forest green or a bright lime?", 'extras': {'signature': 'EnEKbwFpFH0Titgf6hTuQnaeuIgXYPJsSIRx5wCsLCaWxL69de/0TKbzbtVFRU03VsHDmF28TUzjcUH/sr99YjfgAeaB6EHBqv820iAORHzgA4SOXLinMkjOmYcnUwC88T0YUmc/6FuDxfbIDlsS920GHg=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d554-923a-7ed2-aca5-1b0c5a034f52-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 59, 'total_tok

**What a thread id is for:** it separates conversations. One agent can serve many users, as long as each conversation has its own thread id. If two users shared one thread id, they would read each other's memory, so understand this before you build anything with more than one user.

**Limit of `InMemorySaver`:** the snapshots live in RAM, so they are lost when the notebook restarts. For a real application, use a checkpointer backed by a database (for example SQLite or Postgres). The agent code stays the same.

## Part 4: Beyond messages, a custom state
By default the state tracks the **messages** only. You can add your own fields, for instance a user id, to carry other information across the conversation.

> **`AgentState`**: the default state (it holds `messages`). Subclass it to add fields.  **`state_schema=`**: the argument of `create_agent` that tells the agent to use your state class.

In [13]:
from langchain.agents import AgentState

class CustomState(AgentState):
    user_id: str          # my own field, saved with the messages

agent_custom = create_agent(
    model=MODEL,
    state_schema=CustomState,
    checkpointer=InMemorySaver(),
)

config_3 = {"configurable": {"thread_id": "3"}}

response = agent_custom.invoke(
    {"messages": [HumanMessage("Hello my name is Fatima and my favourite colour is green")],
     "user_id": "user_001"},
    config_3,
)

print("user_id in the state:", response["user_id"])

user_id in the state: user_001


In [14]:
# Second run of the same thread: I do NOT pass user_id again
response = agent_custom.invoke(
    {"messages": [HumanMessage("What's my favourite colour?")]},
    config_3,
)

print("user_id still there :", response["user_id"])
print("Answer              :", response["messages"][-1].text)

user_id still there : user_001
Answer              : Your favourite colour is **green**!


## What I learned
1. **Memory is not a property of the model.** The model forgets everything after a run. What remembers is the agent's **state**, and only if it is saved.
2. **State is the list of messages.** By default it is not saved between runs, so every run starts from nothing.
3. **A checkpointer saves a snapshot of the state** at the end of every run. `InMemorySaver` plus `checkpointer=` is the whole mechanism.
4. **A thread id groups runs into one conversation.** Same id: the agent remembers. New id: a fresh start, and the old thread is still there.
5. **Check the message list, not only the reply.** After a second run the list holds four messages: the first exchange is loaded from the snapshot, then the new one is added.
6. **The state can carry more than messages.** Subclass `AgentState` and pass `state_schema=` to keep fields such as a user id.
7. **A model, tools and memory** are the three parts of an agent (notebooks 1_1, 1_2 and 1_3). Everything after this is refinement.

**Mistakes to avoid:**
- Forgetting to pass `config` in the second run: no thread id means no memory, even with a checkpointer.
- Sharing one thread id between two users: they would see each other's conversation.
- Expecting `InMemorySaver` to survive a restart: it lives in RAM.
- Never share API keys (revoke and recreate any key that leaks); run the cells from the top; in Colab the `.env` file on your PC is not visible, use Secrets or paste the key.

## Summary of functions
| Function / class | Simple definition |
|---|---|
| `create_agent(model)` | Builds an agent. Without a checkpointer it forgets after every run. |
| `InMemorySaver()` | Checkpointer that keeps the snapshots in RAM (lost on restart). |
| `create_agent(model, checkpointer=...)` | Builds an agent that saves a snapshot of its state after every run. |
| `config = {"configurable": {"thread_id": "..."}}` | Names the conversation. Pass it as the second argument of `invoke`. |
| `agent.invoke({"messages": [...]}, config)` | Runs the agent inside a thread: earlier messages are loaded, new ones are added. |
| `response["messages"]` | The full message list of the thread, including the earlier runs. |
| `agent.get_state(config)` | Reads the saved snapshot of a thread (`.values["messages"]`). |
| `AgentState` | The default state (messages only). Subclass it to add your own fields. |
| `create_agent(model, state_schema=...)` | Builds an agent that uses your custom state class. |
| `HumanMessage` | A message written by the user. |
| `show()`, `show_messages()`, `run()` | Helpers written in this notebook to print a titled text, every message of a list, and one question with its answer. |

By
> Kassouar Fatima                   24-09-2026

